# Featurizer Tutorial: Temporal (As-Of) Joins

This notebook demonstrates temporal joins in Featurizer using a healthcare scenario:
- **Patients** (target entity) with admission dates
- **Care Plans** (related entity) with plan dates

The key concept is the **as-of join**: for each patient row, we want features from the most recent care plan *as of* that patient's admission date.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# This tutorial is database-free: it loads the config and inspects the
# synthesized features and the generated SQL — none of which touch a
# database. To actually *execute* the features against PostgreSQL, run the
# example script instead (`just example <NN>`, or create_data.py +
# run_example.py with DATABASE_URL / PG* set). See the example README.
sys.path.insert(0, str(Path.cwd().parent.parent))

## 2. Understanding Temporal Relationships

The configuration defines a temporal relationship with `mode: as_of`:

In [2]:
with open("config.yaml") as f:
    print(f.read())

# Healthcare feature generation: Patients with temporal Care Plans

target: patients
max_depth: 2

intervals:
  - P30D  # Last 30 days
  - P90D  # Last 90 days

# A focused primitive set (the full default set would exceed PostgreSQL's 1664
# columns-per-row limit on this config). Includes the rolling ordered-set stats
# — rolling_median / rolling_iqr render as correlated subqueries (PostgreSQL
# forbids OVER on percentile_cont), exercised end to end here.
aggregations:
  - count
  - sum
  - mean
  - min
  - max
  - stddev
  - median
transformations:
  - identity
  - abs
  - lag_1
  - rolling_mean_7
  - rolling_median_7
  - rolling_iqr_7

entities:
  - alias: patients
    id: patient_id
    table: patients
    temporal_ix: admission_date
    variables:
      age:
        type: numeric
      severity_level:
        type: categorical

  - alias: care_plans
    id: plan_id
    table: care_plans
    temporal_ix: plan_date
    variables:
      treatment_type:
        type: categorical
      

### Key Temporal Configuration:

```yaml
temporal:
  mode: as_of        # Enable as-of join semantics
  grace: P7D         # Allow matching up to 7 days in the future
  child_time: plan_date  # Override child entity's temporal column
```

This means: for each patient admission, find the most recent care plan where `plan_date <= admission_date + 7 days`.

## 3. Load Configuration

In [3]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target: {featurizer.target.alias}")
print(f"Intervals: {featurizer.intervals}")

2026-06-21 13:29:15.899 | DEBUG    | featurizer.planner:plan:230 - Starting feature build for target patients


2026-06-21 13:29:15.900 | DEBUG    | featurizer.planner:_build_features:256 - build_features(patients) depth=0


2026-06-21 13:29:15.900 | DEBUG    | featurizer.planner:_build_features:256 - build_features(care_plans) depth=1


2026-06-21 13:29:15.900 | INFO     | featurizer.planner:_build_features:276 - Maximum recursion depth reached at depth 2; materializing care_plans without traversing further.


2026-06-21 13:29:15.900 | DEBUG    | featurizer.planner:_build_aggregations:1014 - Processing backward relationship Entity(patients).patient_id -> Entity(care_plans).patient_id


2026-06-21 13:29:15.901 | WARNING  | featurizer.planner:_apply_direct_roles:1155 - Direct variable 'patients.severity_level' (type: categorical) will pass through as a raw string column and is likely to crash a downstream encoder. Set role: categorical (with a declared vocabulary or a PostgreSQL ENUM) to one-hot encode it, or role: identifier to exclude it.


Target: patients
Intervals: ['P30D', 'P90D']


In [4]:
# Examine the temporal relationship
for rel in featurizer.relationships:
    print(f"Relationship: {rel.parent.alias} -> {rel.child.alias}")
    print(f"  Temporal mode: {rel.temporal_mode}")
    print(f"  Grace period: {rel.temporal_grace}")
    print(f"  Child timestamp: {rel.temporal_child_field}")

Relationship: patients -> care_plans
  Temporal mode: as_of
  Grace period: P7D
  Child timestamp: None


## 4. Examining the Generated SQL

The key difference from regular joins is the `LEFT JOIN LATERAL` clause.

In [5]:
sql = featurizer.query

# Find the LATERAL join
if "lateral" in sql.lower():
    print("Temporal join generates LEFT JOIN LATERAL:")
    print("=" * 60)

    # Extract the lateral join portion
    lines = sql.split("\n")
    in_lateral = False
    for line in lines:
        if "lateral" in line.lower():
            in_lateral = True
        if in_lateral:
            print(line)
            if ") as" in line.lower() and "on true" in line.lower():
                break

2026-06-21 13:29:15.922 | DEBUG    | featurizer.sql:render:40 - Rendered SQL for target 'patients': 5 CTEs, 248797 chars


Temporal join generates LEFT JOIN LATERAL:
        cross join lateral (

        with

        
        -- sythetize aggregations and direct features for care_plans
        care_plans_synth as (
        select
        care_plans.plan_id, care_plans.plan_date, care_plans.patient_id, cost, treatment_type
        from care_plans
        
        
        )
        ,
        -- transform care_plans
        care_plans_transform as (
        select
        plan_id, plan_date, patient_id,  abs(cost)  as "ABS(care_plans.cost)" , lag(cost, 1) over (partition by plan_id order by plan_date) as "LAG_1(care_plans.cost)" , lag(treatment_type, 1) over (partition by plan_id order by plan_date) as "LAG_1(care_plans.treatment_type)" , ((select percentile_cont(0.75) within group (order by _w.v) from (select care_plans_synth.cost as v from care_plans_synth where care_plans_synth.plan_id = _ego.plan_id and care_plans_synth.plan_date <= _ego.plan_date order by care_plans_synth.plan_date desc limit 7) _w)) -

### Understanding the LATERAL Join

The generated SQL uses a pattern like:

```sql
LEFT JOIN LATERAL (
    SELECT care_plans_transform.cost, ...
    FROM care_plans_transform
    WHERE care_plans_transform.patient_id = patients.patient_id
      AND care_plans_transform.plan_date <= patients.admission_date
      AND patients.admission_date - care_plans_transform.plan_date <= interval 'P7D'
    ORDER BY care_plans_transform.plan_date DESC
    LIMIT 1
) AS care_plans_asof_for_patients ON true
```

This ensures:
1. Only matching patient_id
2. Care plan date is before or at admission date
3. Within the grace period
4. Most recent match (ORDER BY ... DESC LIMIT 1)

## 5. Generated Features

In [6]:
target_features = featurizer.features[featurizer.target.alias]

print(f"Total features: {len(target_features)}")
print("\nFeatures from care_plans (via temporal join):")
care_plan_features = [f for f in target_features if "care_plans" in f.name.lower()]
for f in sorted(care_plan_features, key=lambda x: x.name)[:15]:
    print(f"  {f.name}")

Total features: 730

Features from care_plans (via temporal join):
  "ABS(patients.COUNT(care_plans.LAG_1(care_plans.treatment_type)))"
  "ABS(patients.COUNT(care_plans.LAG_1(care_plans.treatment_type)|inte~4f9badff)"
  "ABS(patients.COUNT(care_plans.LAG_1(care_plans.treatment_type)|inte~d7382445)"
  "ABS(patients.COUNT(care_plans.plan_date))"
  "ABS(patients.COUNT(care_plans.plan_date|interval=P30D))"
  "ABS(patients.COUNT(care_plans.plan_date|interval=P90D))"
  "ABS(patients.COUNT(care_plans.plan_id))"
  "ABS(patients.COUNT(care_plans.plan_id|interval=P30D))"
  "ABS(patients.COUNT(care_plans.plan_id|interval=P90D))"
  "ABS(patients.COUNT(care_plans.treatment_type))"
  "ABS(patients.COUNT(care_plans.treatment_type|interval=P30D))"
  "ABS(patients.COUNT(care_plans.treatment_type|interval=P90D))"
  "ABS(patients.MAX(care_plans.ABS(care_plans.cost)))"
  "ABS(patients.MAX(care_plans.ABS(care_plans.cost)|interval=P30D))"
  "ABS(patients.MAX(care_plans.ABS(care_plans.cost)|interval=P90D))"


## 6. When to Use Temporal Joins

Use temporal (as-of) joins when:

1. **Point-in-time correctness matters**: You need features as they would have been at a specific moment
2. **Preventing data leakage**: In ML, you can't use future data to predict past events
3. **Time-varying relationships**: The "current" related record changes over time

### Examples:
- Customer's current address at time of order
- Patient's latest diagnosis at time of treatment
- Product's price at time of purchase
- Employee's department at time of performance review

In [7]:
print("Summary")
print("=" * 40)
print(f"Target: {featurizer.target.alias}")
print(f"Temporal relationship: patients <- care_plans (as-of)")
print(f"Grace period: P7D")
print(f"Total features: {len(target_features)}")

Summary
Target: patients
Temporal relationship: patients <- care_plans (as-of)
Grace period: P7D
Total features: 730
